# **Model Evaluation**

## Objectives

* Load the trained model
* Evaluate performance
* Calculate classification accuracy and loss
* Analyse predictions using a confusion matrix and classification report 
* Determine whether the model meets 97% accuracy rate

## Inputs

* Trained model
* Test images

## Outputs

* Test accuracy and loss 
* Confusion matrix 
* Precision recall and F1 score
* Evaluation findings for the Streamlit dashboard

## Additional Comments

* The test dataset has not been used during model training or model selection. It provides an independent assessment of model performance.



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path("/workspaces/FInal-Project-01")

os.chdir(PROJECT_ROOT)

print("Current working directory:", Path.cwd())

Current working directory: /workspaces/FInal-Project-01


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score
)

print("TensorFlow version:", tf.__version__)
print("Current directory:", Path.cwd())

2026-10-14 20:37:24.781550: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-14 20:37:33.484850: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version: 2.16.1
Current directory: /workspaces/FInal-Project-01


In [3]:
model_path = Path("outputs/models/cherry_leaf_cnn.keras")

model = tf.keras.models.load_model(model_path)

print("Model loaded successfully!")
print("Model input shape:", model.input_shape)

Model loaded successfully!
Model input shape: (None, 100, 100, 3)


In [4]:
test_path = Path("data/processed/test")

IMAGE_SIZE = (100, 100)
BATCH_SIZE = 32

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_path,
    labels="inferred",
    label_mode="binary",
    class_names=["healthy", "powdery_mildew"],
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False 
)

print("Class names:", test_dataset.class_names)

Found 646 files belonging to 2 classes.
Class names: ['healthy', 'powdery_mildew']


In [5]:
test_loss, test_accuracy = model.evaluate(
    test_dataset,
    verbose=1
)

print(f"Test Accuracy: {test_accuracy:.2%}")
print(f"Test Loss: {test_loss:.4f}")

target_accuracy = 0.97

if test_accuracy >= target_accuracy:
    print("The model meets the 97% accuracy target.")
else:
    print("The model does not yet meet the 97% accuracy target.")

 1/21 ━━━━━━━━━━━━━━━━━━━━ 17s 886ms/step - accuracy: 1.0000 - loss: 9.6465e-05

21/21 ━━━━━━━━━━━━━━━━━━━━ 2s 59ms/step - accuracy: 0.9969 - loss: 0.0289
Test Accuracy: 99.69%
Test Loss: 0.0289
The model meets the 97% accuracy target.


---

# Confusion Matrix

A confusion matrix is used to examine the model's predictions
for each class.

This helps identify whether the model is more likely to
misclassify healthy leaves or leaves affected by powdery mildew.

In [6]:
# Collect the actual labels
y_true = np.concatenate([
    labels.numpy().flatten()
    for images, labels in test_dataset
]).astype(int)

# Generate predicted probabilities
y_probabilities = model.predict(test_dataset).flatten()

# Convert probabilities into class predictions
y_pred = (y_probabilities >= 0.5).astype(int)

# Create confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Healthy", "Powdery Mildew"],
    yticklabels=["Healthy", "Powdery Mildew"]
)

plt.title("Confusion Matrix — Cherry Leaf CNN")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.tight_layout()
plt.show()

 1/21 ━━━━━━━━━━━━━━━━━━━━ 2s 149ms/step

2026-10-14 20:37:45.143989: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


21/21 ━━━━━━━━━━━━━━━━━━━━ 1s 59ms/step


<Figure size 700x500 with 2 Axes>

In [7]:
print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=["Healthy", "Powdery Mildew"],
        digits=4
    )
)

Confusion Matrix:
[[330   0]
 [  2 314]]

Classification Report:
                precision    recall  f1-score   support

       Healthy     0.9940    1.0000    0.9970       330
Powdery Mildew     1.0000    0.9937    0.9968       316

      accuracy                         0.9969       646
     macro avg     0.9970    0.9968    0.9969       646
  weighted avg     0.9969    0.9969    0.9969       646



In [8]:
from pathlib import Path
import json

evaluation_path = Path("outputs/evaluation")
evaluation_path.mkdir(parents=True, exist_ok=True)

# Save overall model performance
metrics = {
    "test_accuracy": float(test_accuracy),
    "test_loss": float(test_loss),
    "target_accuracy": 0.97,
    "target_met": bool(test_accuracy >= 0.97),
    "test_images": int(len(y_true))
}

with open(evaluation_path / "test_metrics.json", "w") as file:
    json.dump(metrics, file, indent=4)

# Save the classification report
report = classification_report(
    y_true,
    y_pred,
    target_names=["Healthy", "Powdery Mildew"],
    output_dict=True
)

pd.DataFrame(report).transpose().to_csv(
    evaluation_path / "classification_report.csv"
)

# Save the confusion matrix
pd.DataFrame(
    cm,
    index=["Actual Healthy", "Actual Powdery Mildew"],
    columns=["Predicted Healthy", "Predicted Powdery Mildew"]
).to_csv(evaluation_path / "confusion_matrix.csv")

print("Evaluation results saved successfully!")

Evaluation results saved successfully!


## Model Evaluation Findings

The CNN achieved a test accuracy of **99.69%**, exceeding
the client's minimum requirement of 97%.

### Key Results
- Test images evaluated: 646
- Correct classifications: 644
- Incorrect classifications: 2
- Test accuracy: 99.69%
- Test loss: 0.0289
- Healthy recall: 100%
- Powdery mildew recall: 99.37%

### Interpretation
The model successfully distinguished healthy leaves from
powdery mildew in nearly all test images.

Both incorrect predictions were powdery mildew leaves
classified as healthy (false negatives).

This is an important limitation because failing to identify
a diseased leaf could delay appropriate action.

### Business Requirement
The model exceeds the required 97% classification accuracy
on the held-out test dataset.

### Limitations
The dataset contains augmented images. Source-image grouping
was used during splitting to reduce data leakage.

Further checks for duplicate images and evaluation on new,
independently collected photographs would strengthen
confidence in the model's generalisation

## Data Leakage Check

The dataset was split using source-image groups to keep known
augmented versions of each photograph together.

An additional SHA-256 hash check is performed to identify
identical image files appearing across training, validation,
and test datasets.

This check detects exact file duplicates but does not detect
visually identical images saved with different compression,
rotations, or other transformations.

In [9]:
import hashlib
from collections import defaultdict
from pathlib import Path

processed_path = Path("data/processed")

split_names = ["train", "validation", "test"]
supported_extensions = {".jpg", ".jpeg", ".png", ".webp"}

# Store which dataset splits contain each image hash
hash_locations = defaultdict(set)

for split_name in split_names:
    split_folder = processed_path / split_name

    for image_path in split_folder.rglob("*"):
        if (
            image_path.is_file()
            and image_path.suffix.lower() in supported_extensions
        ):
            image_hash = hashlib.sha256(
                image_path.read_bytes()
            ).hexdigest()

            hash_locations[image_hash].add(split_name)

# Identify identical files appearing in different splits
cross_split_duplicates = {
    image_hash: locations
    for image_hash, locations in hash_locations.items()
    if len(locations) > 1
}

print("Unique file hashes:", len(hash_locations))
print(
    "Exact duplicate hashes across splits:",
    len(cross_split_duplicates)
)

Unique file hashes: 4208
Exact duplicate hashes across splits: 0


---

## Conclusions and Next Steps

### Conclusions

- The trained CNN was evaluated using the held-out test dataset containing 646 cherry leaf images.
- The model achieved **99.69% test accuracy**, exceeding the client's minimum requirement of **97%**.
- Of the 646 test images, 644 were classified correctly and 2 were classified incorrectly.
- Both incorrect predictions were powdery mildew leaves classified as healthy.
- The confusion matrix and classification report were used to assess performance for each category.
- Exact duplicate image files were checked across the training, validation and test datasets using SHA-256 hashes.
- Source-image grouping during data preparation helped reduce the risk of data leakage, although it cannot guarantee that all visually related images were identified.
- The trained model meets the project's classification accuracy requirement on the held-out test dataset.

### Next Steps

- Use the saved model in the Streamlit dashboard to classify uploaded cherry leaf images.
- Display the predicted category and confidence score to the user.
- Present the model evaluation results in the dashboard's Model Performance page.
- Test the dashboard's prediction workflow using healthy and powdery mildew images.
- Consider evaluating the model on new, independently collected photographs to assess its performance beyond the project dataset.